<a href="https://colab.research.google.com/github/brunoossouza/Limpeza-Dados-E-Commerce-Spark/blob/main/LimpezaDados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark


In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.getOrCreate()

In [ ]:
produtos = spark.read.csv("drive/MyDrive/produtos.csv", header=True, inferSchema=True)
vendedores = spark.read.csv("drive/MyDrive/vendedores.csv",header=True, inferSchema=True)
clientes = spark.read.csv("drive/MyDrive/clientes.csv",header=True, inferSchema=True)
itens_pedidos = spark.read.csv("drive/MyDrive/itens_pedido.csv",header=True, inferSchema=True)
pagamentos_pedido = spark.read.csv("drive/MyDrive/pagamentos_pedido.csv",header=True, inferSchema=True)
avaliacoes_produto = spark.read.csv("drive/MyDrive/avaliacoes_pedido.csv",header=True, inferSchema=True)
pedidos = spark.read.csv("drive/MyDrive/pedidos.csv",header=True, inferSchema=True)

In [ ]:
print('dataframe produtos: \n', produtos.show(n=5, truncate=False))
print('dataframe vendedores: \n', vendedores.show(5))
print('dataframe clientes: \n', clientes.show(5))
print('dataframe itens_pedidos: \n', itens_pedidos.show(5))
print('dataframe pagamentos_pedido: \n', pagamentos_pedido.show(5))
print('dataframe avaliacoes_pedidos: \n', avaliacoes_produto.show(5))
print('dataframe pedidos:\n,', pedidos.show(5))

+--------------------------------+---------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|id_produto                      |categoria_produto    |tamanho_nome_produto|tamanho_descricao_produto|quantidade_fotos_produto|peso_produto_g|comprimento_produto_cm|altura_produto_cm|largura_produto_cm|
+--------------------------------+---------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                  |287                      |1                       |225           |16                    |10               |14                |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                  |276                      |1                       |1000          |30                    |18               |

In [ ]:
# Acessar coluna
from pyspark.sql.functions import col

clientes.select('id_cliente').show(1, truncate=False)
clientes.select(col('id_cliente')).show(1)
clientes.select(clientes['id_cliente']).show(1)
clientes.select(clientes.id_cliente).show(1)

+--------------------------------+
|id_cliente                      |
+--------------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|
+--------------------------------+
only showing top 1 row
+--------------------+
|          id_cliente|
+--------------------+
|06b8999e2fba1a1fb...|
+--------------------+
only showing top 1 row
+--------------------+
|          id_cliente|
+--------------------+
|06b8999e2fba1a1fb...|
+--------------------+
only showing top 1 row
+--------------------+
|          id_cliente|
+--------------------+
|06b8999e2fba1a1fb...|
+--------------------+
only showing top 1 row


In [ ]:
produtos_tratar_categ_nulos = produtos.na.fill({'categoria_produto': 'Não especificado'})

produtos_tratar_categ_nulos.filter(col('categoria_produto')=='Não especificado').count()

610

In [ ]:
print('Total de pedidos', pedidos.count())

pedidos_unicos = pedidos.dropDuplicates()
print('Total de pedidos unicos: ', pedidos_unicos.count())

pedidos_remocao_nulos = pedidos_unicos.na.drop()
print('Total de pedidos após limpeza fr valores nulos:', pedidos_remocao_nulos.count())

pedidos_remocao_nulos_id = pedidos_unicos.na.drop(subset=['id_cliente', 'id_pedido'])
print('Total de pedidos após limpeza de id nulos:', pedidos_remocao_nulos_id.count())

Total de pedidos 99441
Total de pedidos unicos:  99441
Total de pedidos após limpeza fr valores nulos: 96461
Total de pedidos após limpeza de id nulos: 99441


In [ ]:
colunas = ["peso_produto_g", "comprimento_produto_cm", "altura_produto_cm", "largura_produto_cm"]

for coluna in colunas:
  produtos = produtos.na.fill({coluna: 0})

In [ ]:
produtos.write.mode('overwrite').option('header','true').csv('drive/MyDrive/produtos_tratada.csv')

In [ ]:
spark.read.option('header', 'true').csv('drive/MyDrive/produtos_tratada.csv').show(5)

+--------------------+--------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|          id_produto|   categoria_produto|tamanho_nome_produto|tamanho_descricao_produto|quantidade_fotos_produto|peso_produto_g|comprimento_produto_cm|altura_produto_cm|largura_produto_cm|
+--------------------+--------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|1e9e8ef04dbcff454...|          perfumaria|                  40|                      287|                       1|           225|                    16|               10|                14|
|3aa071139cb16b67c...|               artes|                  44|                      276|                       1|          1000|                    30|               18|                20|
|96bd76ec8810374ed...|       esporte_lazer|  